In [5]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
from helpers import Trial
from helpers.metadata import PIXEL_RAILWAY_WIDTH, PIXEL_RAIL_TOP, PIXEL_RAIL_BOTTOM

In [ ]:
DATA_FOLDER = "preprocessed"
PHASE_ORDER = [("Pre-Test", "reaching"), 
               ("Pre-Test", "avoiding"), 
               ("Training", None),
               ("Post-Test", "reaching"), 
               ("Recap", "avoiding"),  
               ("Post-Test", "avoiding")]

#  Plot trajectories

In [7]:
RAIL_WIDTH_CM = Trial._transform_px_to_cm(PIXEL_RAILWAY_WIDTH, 0)
RAIL_HEIGHT_CM = Trial._transform_px_to_cm(PIXEL_RAIL_TOP - PIXEL_RAIL_BOTTOM, 1)

In [8]:
for cond_path in ["ai", "ad", "ri", "rd"]:
    cond = pd.read_csv(f"{DATA_FOLDER}/{cond_path}.csv")
    cond_rel = pd.read_csv(f"{DATA_FOLDER}/{cond_path}_relevant_trials.csv")

    for participant_id in cond["participant_id"].unique():

        sys.stdout.write(f"\rProcessing {participant_id}")

        figure = plt.figure(figsize=(25, 65))

        mapping = cond["mapping"].iloc[0]
        natural = cond["natural_mapping"].iloc[0]

        participant_trials = cond[(cond["participant_id"] == participant_id)]
        subfigures = figure.subfigures(participant_trials.groupby(["phase", "block", "task"]).ngroups, 1)
        subfig_nr = 0

        for phase, task in PHASE_ORDER:

            phase_task_trials = None
            if task == None:
                phase_task_trials = participant_trials[(participant_trials["phase"] == phase)]
            else:
                phase_task_trials = participant_trials[(participant_trials["phase"] == phase) & (participant_trials["task"] == task)]

            for block in sorted(phase_task_trials["block"].unique()):
                
                block_trials = phase_task_trials[phase_task_trials["block"] == block]
                natural = participant_trials["natural_mapping"].iloc[0]
                for task, task_trials in block_trials.groupby(["task"]):
                    
                    axs = subfigures[subfig_nr].subplots(1, len(block_trials["target_pos_y_cm"].unique()))
                    subfigures[subfig_nr].suptitle(f"{block}[{phase}] - {task}")

                    axis_nr = 0
                    targets = sorted(task_trials["target_pos_y_cm"].unique())
                    for distance in targets:
                        distance_trials = task_trials[task_trials["target_pos_y_cm"] == distance]

                        axs[axis_nr].set_title(f"{distance}")
                        axs[axis_nr].axhline(distance, linestyle="--", color="gray")

                        axs[axis_nr].set_xlim(-1, RAIL_WIDTH_CM + 1)
                        axs[axis_nr].set_ylim(-1, RAIL_HEIGHT_CM + 1)
                        axs[axis_nr].axvline(-0.1, color="darkgray")
                        axs[axis_nr].axvline(RAIL_WIDTH_CM+0.1, color="darkgray")

                        for trial_index, trial in distance_trials.groupby(["trial_index"]):
                            axs[axis_nr].plot(trial["current_pos_x_cm"], trial["current_pos_y_cm"], c="k", alpha=0.15)

                            identified_relevant = cond_rel[(cond_rel["participant_id"] == participant_id) & (cond_rel["trial_index"] == trial_index)]
                            axs[axis_nr].scatter(identified_relevant["current_pos_x_cm"].iloc[0], identified_relevant["current_pos_y_cm"].iloc[0], marker="x" , linewidths=1, c="k")

                        axis_nr += 1

                    subfig_nr += 1

        plt.savefig(f"{DATA_FOLDER}/figures/{participant_id}_{mapping}_natural_{natural}")
        plt.close()

Processing 9rdd